[source](https://github.com/rasbt/LLMs-from-scratch/blob/main/ch05/07_gpt_to_llama/converting-gpt-to-llama2.ipynb)

In [1]:
%matplotlib inline

import os
import sys
import math

from sys import platform

sys.path.append('../../')

import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from importlib.metadata import version 

import tiktoken

import torch
import torch.nn as nn



%load_ext autoreload
%autoreload 2

from llm_from_scratch.CH4.gpt import GPTModel
from llm_from_scratch.CH4.llama import Llama2Model
from llm_from_scratch.CH4.blocks import RMSNorm, SiLU, MultiheadAttentionLlama
from llm_from_scratch.CH4.utils import precompute_rope_params, compute_rope, calc_model_memory_size
from llm_from_scratch.CH2.text_data_set import create_dataloader_v1
from llm_from_scratch.CH5.loss import calc_loss_batch
from llm_from_scratch.CH5.utils import find_highest_gradient, load_weights_into_llama
from llm_from_scratch.CH5.optim import evaluate_model, generate_and_print_sample
from llm_from_scratch.CH5.utils import token_ids_to_text, text_to_token_ids, generate
print(f"torch version: {version('torch')}")

torch version: 2.12.0


In [2]:
from importlib.metadata import version

pkgs=["huggingface_hub", # to download pretrained weights
      "sentencepiece",   # to implement tokenizer
      "torch"            # to implement model
      ]
for p in pkgs: print(f"{p} version: {version(p)}")

huggingface_hub version: 1.14.0
sentencepiece version: 0.2.1
torch version: 2.12.0


In [3]:
torch.manual_seed(123)
example_batch=torch.rand(2,3,4)
rms_norm=RMSNorm(emb_dim=example_batch.shape[-1], eps=1e-5)
rmsnorm_pytorch=torch.nn.RMSNorm(example_batch.shape[-1], eps=1e-5)
assert torch.allclose(rms_norm(example_batch), rmsnorm_pytorch(example_batch))

silu=SiLU()
assert torch.allclose(silu(example_batch), torch.nn.functional.silu(example_batch))

### RoPE

In [4]:
# apply RoPE to q and k
# settings
batch_size=2
context_len=5
num_heads=2
head_dim=16
# instantiate RoPE parameters
cos, sin=precompute_rope_params(head_dim=head_dim, context_length=context_len)
# dummy query and key tensors
torch.manual_seed(123)
queries=torch.rand(batch_size, num_heads, context_len, head_dim)
keys=torch.rand(batch_size, num_heads, context_len, head_dim)
# apply rotary position embeddings
queries_ro=compute_rope(queries, cos, sin)
keys_rot=compute_rope(keys, cos, sin)

### MultiheadAttention

In [5]:
# settings
batch_size=2
context_len=100
max_context_len=4096
embed_dim=128
num_heads=4

example_batch=torch.rand(batch_size, context_len, embed_dim)
mha=MultiheadAttentionLlama(d_in=embed_dim, d_out=embed_dim, context_length=max_context_len, num_heads=num_heads)
outs=mha(example_batch)
print(f'{outs.shape=}')
del mha, outs

outs.shape=torch.Size([2, 100, 128])


### Llama2

In [6]:
LLAMA2_CONFIG_7B={
    "vocab_size":32000,    # vocabulary size
    "context_length":4096, # context length
    "emb_dim":4096,        # embedding dimension
    "n_heads":32,          # number of attention heads
    "n_layers":32,         # number of layers
    "hidden_dim":11008,    # size of intermediate dimension in FeedForward
    "dtype":torch.bfloat16 # lower precision dtype to reduce memory usage
}
model=Llama2Model(LLAMA2_CONFIG_7B)
total_params=sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")
print(f"Model size in float32 (default): {calc_model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"Model size in bfloat16 (default): {calc_model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

Total number of parameters: 6,738,415,616
Model size in float32 (default): 52.33 GB
Model size in bfloat16 (default): 26.17 GB


In [7]:
device=torch.device('cuda') if torch.cuda.is_available() else torch.device('mps') if torch.mps.is_available() else torch.device('cpu')
model.to(device);

### Tokenizer

In [8]:
hugging_face_token=

In [9]:
from huggingface_hub import login

login(token=hugging_face_token)

/Users/reaungamornrat.sureerat/miniforge3/envs/llm/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
from huggingface_hub import hf_hub_download

tokenizer_file=hf_hub_download(repo_id="meta-llama/Llama-2-7b", filename="tokenizer.model",
                               local_dir="/Users/reaungamornrat.sureerat/data/Llama2")

In [11]:
import sentencepiece as spm

class LlamaTokenizer:
    def __init__(self, tokenizer_file):
        sp=spm.SentencePieceProcessor()
        sp.load(tokenizer_file)
        self.tokenizer=sp
    def encode(self, text, allowed_special=None): return self.tokenizer.encode(text, out_type=int)
    def decode(self, ids): return self.tokenizer.decode(ids)

tokenizer=LlamaTokenizer(tokenizer_file)

torch.manual_seed(123)
token_ids=generate(model=model, idx=text_to_token_ids("Every effort moves", tokenizer, allowed_special=None).to(device), 
                   max_new_tokens=30, context_size=LLAMA2_CONFIG_7B['context_length'], top_k=1, temperature=0.)
print(f"Output text: {token_ids_to_text(token_ids, tokenizer)}")

Output text: Every effort moves Atl Bischof nan counted Mant сла reception Bischof doit员napshot др Past鳥lustor tied warniy sobre perpet Application bedIALchmarkBinary Configuration ersten komm Lista


### Load pretrained weight

In [15]:
weights_file=hf_hub_download(repo_id='meta-llama/Llama-2-7b', filename='consolidated.00.pth', 
                             local_dir="/Users/reaungamornrat.sureerat/data/Llama2/Llama-2-7b")
weights=torch.load(weights_file, weights_only=True)
print(f"{list(weights.keys())[:15]=}")
load_weights_into_llama(model, LLAMA2_CONFIG_7B, weights)
model.to(device);

list(weights.keys())[:15]=['tok_embeddings.weight', 'norm.weight', 'output.weight', 'layers.0.attention.wq.weight', 'layers.0.attention.wk.weight', 'layers.0.attention.wv.weight', 'layers.0.attention.wo.weight', 'layers.0.feed_forward.w1.weight', 'layers.0.feed_forward.w2.weight', 'layers.0.feed_forward.w3.weight', 'layers.0.attention_norm.weight', 'layers.0.ffn_norm.weight', 'layers.1.attention.wq.weight', 'layers.1.attention.wk.weight', 'layers.1.attention.wv.weight']


In [16]:
# generate text
torch.manual_seed(123)
token_ids=generate(model=model, idx=text_to_token_ids("Every effort", tokenizer, allowed_special=None).to(device),
                   max_new_tokens=25, context_size=LLAMA2_CONFIG_7B["context_length"], top_k=1, temperature=0.)
print(f"Output text: {token_ids_to_text(token_ids, tokenizer)}")

Output text: Every effort has been made to ensure that the information contained in this website is accurate and up to date. However, the information is provided


### Instruction-finetuned model

A model capable of following instruction

In [17]:
del model # free up memory

weight_file=hf_hub_download(repo_id="meta-llama/Llama-2-7b-chat", filename='consolidated.00.pth', 
                            local_dir="/Users/reaungamornrat.sureerat/data/Llama2/Llama-2-7b-chat")
weights=torch.load(weight_file, weights_only=True)
model=Llama2Model(LLAMA2_CONFIG_7B)
load_weights_into_llama(model, LLAMA2_CONFIG_7B, weights)
model.to(device);

torch.manual_seed(123)
token_ids=generate(model=model, idx=text_to_token_ids("What do llamas eat?", tokenizer, allowed_special=None).to(device),
                  max_new_tokens=25, context_size=LLAMA2_CONFIG_7B['context_length'], top_k=1, temperature=0.)
print("Output text: ", token_ids_to_text(token_ids, tokenizer))

Output text:  What do llamas eat?

Llamas are herbivores, which means they eat plants for their food. They feed on a variety
